# 2. Entrenamiento y evaluación de modelos

Entrenamos cuatro clasificadores distintos sobre el mismo split, con `Pipeline` (preprocesamiento + modelo) y búsqueda de hiperparámetros vía `GridSearchCV` con validación cruzada estratificada de 5 folds.

Modelos:
1. **Random Forest** (`sklearn.ensemble.RandomForestClassifier`)
2. **XGBoost** con soporte GPU — `tree_method='hist'` + `device='cuda'` (API moderna ≥ 2.0; `gpu_hist` está deprecado).
3. **CatBoost** (`CatBoostClassifier`)
4. **LightGBM** (`LGBMClassifier`)

Comparamos por **ROC-AUC** (métrica robusta a desbalance) y reportamos también accuracy, precision, recall, F1, matriz de confusión y curva ROC. Al final guardamos el mejor pipeline en `app/model.joblib` para que la API lo cargue.

In [ ]:
import warnings, time, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report,
)

# Importamos el preprocesador desde el módulo del proyecto. Esto garantiza
# que entrenamiento y serving usen exactamente la misma lógica.
import sys
sys.path.append("..")
from app.preprocessing import (
    ALL_FEATURES, TARGET_COLUMN,
    build_preprocessor, build_feature_frame, clean_total_charges,
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
USE_GPU = True   # cambia a False si no tienes GPU NVIDIA

## 2.1 Carga y split

In [ ]:
df = pd.read_csv("../data/telco_churn.csv").drop(columns=["customerID"])
df = clean_total_charges(df)

y = (df[TARGET_COLUMN] == "Yes").astype(int)
X = build_feature_frame(df.drop(columns=[TARGET_COLUMN]))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE,
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Churn rate train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

**Análisis**: el `stratify=y` mantiene proporciones equivalentes en ambos lados — necesario porque el desbalance es relevante. La validación cruzada también será estratificada.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2.2 Función de evaluación común

In [ ]:
def evaluate(model, X_te, y_te, name):
    """Calcula métricas y devuelve dict + imprime resumen."""
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    metrics = {
        "model": name,
        "accuracy":  accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred),
        "recall":    recall_score(y_te, y_pred),
        "f1":        f1_score(y_te, y_pred),
        "roc_auc":   roc_auc_score(y_te, y_proba),
    }
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        if k != "model":
            print(f"  {k:>10}: {v:.4f}")
    return metrics, y_proba

## 2.3 Random Forest

In [ ]:
rf_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced",
    )),
])
rf_grid = {
    "model__n_estimators":      [200, 400],
    "model__max_depth":         [None, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf":  [1, 2],
    "model__max_features":      ["sqrt"],
    "model__bootstrap":         [True],
    "model__criterion":         ["gini", "entropy"],
}
t0 = time.time()
rf_search = GridSearchCV(rf_pipe, rf_grid, scoring="recall", cv=cv,
                         n_jobs=-1, verbose=1, refit=True)
rf_search.fit(X_train, y_train)
print(f"Tiempo: {time.time()-t0:.1f}s")
print("Mejores params:", rf_search.best_params_)
rf_metrics, rf_proba = evaluate(rf_search.best_estimator_, X_test, y_test, "RandomForest")

**Análisis Random Forest**:
- Buen baseline robusto, no requiere mucho tuning.
- `class_weight='balanced'` reajusta automáticamente para compensar la clase minoritaria — sube el recall a costa de la precision.
- Es el modelo más interpretable del trío de árboles ensamblados (bagging > boosting en simplicidad).

## 2.4 XGBoost (con GPU si está disponible)

In [ ]:
from xgboost import XGBClassifier
xgb_kwargs = dict(
    random_state=RANDOM_STATE, n_jobs=-1,
    eval_metric="logloss", tree_method="hist",
)
if USE_GPU:
    # API moderna xgboost >= 2.0
    xgb_kwargs["device"] = "cuda"
xgb_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", XGBClassifier(**xgb_kwargs)),
])
xgb_grid = {
    "model__n_estimators":     [200, 400],
    "model__max_depth":        [4, 6],
    "model__learning_rate":    [0.05, 0.1],
    "model__subsample":        [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__gamma":            [0, 1],
    "model__min_child_weight": [1, 3],
    "model__reg_alpha":        [0, 0.1],
    "model__reg_lambda":       [1.0],
    "model__scale_pos_weight": [1.0, 2.7],
}
t0 = time.time()
xgb_search = GridSearchCV(xgb_pipe, xgb_grid, scoring="recall", cv=cv,
                          n_jobs=-1, verbose=1, refit=True)
xgb_search.fit(X_train, y_train)
print(f"Tiempo: {time.time()-t0:.1f}s")
print("Mejores params:", xgb_search.best_params_)
xgb_metrics, xgb_proba = evaluate(xgb_search.best_estimator_, X_test, y_test, "XGBoost")

**Análisis XGBoost**:
- En GPU el grid completo termina varias veces más rápido. Para verificar uso de GPU: durante la ejecución abre el Administrador de Tareas (Windows) o `nvidia-smi -l 1` (Linux) y mira el porcentaje del dispositivo.
- `scale_pos_weight=2.7` es la relación negativos/positivos en el train set; le dice al modelo que penalice más fallar en la clase minoritaria.
- **Importante**: si ves un warning como *"WARNING: ... If you are using cudf and dask, ..."* o similar, no afecta la corrección. Si en cambio ves *"GPU device not found"*, el grid se ejecuta en CPU silenciosamente — verifica que `xgboost` esté compilado con CUDA (`pip install xgboost` ya viene con GPU support en la mayoría de plataformas).

## 2.5 CatBoost

In [ ]:
from catboost import CatBoostClassifier
cb_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", CatBoostClassifier(random_seed=RANDOM_STATE, verbose=False)),
])
cb_grid = {
    "model__iterations":          [200, 400],
    "model__depth":               [4, 6],
    "model__learning_rate":       [0.05, 0.1],
    "model__l2_leaf_reg":         [3, 5],
    "model__bagging_temperature": [0, 1],
    "model__border_count":        [128],
    "model__random_strength":     [1],
}
t0 = time.time()
cb_search = GridSearchCV(cb_pipe, cb_grid, scoring="recall", cv=cv,
                         n_jobs=-1, verbose=1, refit=True)
cb_search.fit(X_train, y_train)
print(f"Tiempo: {time.time()-t0:.1f}s")
print("Mejores params:", cb_search.best_params_)
cb_metrics, cb_proba = evaluate(cb_search.best_estimator_, X_test, y_test, "CatBoost")

**Análisis CatBoost**: aunque CatBoost brilla con categóricas nativas (sin one-hot), aquí lo alimentamos ya transformado para mantener todos los modelos sobre el mismo input. Aún así suele competir muy bien y maneja sobreajuste con `l2_leaf_reg` y `bagging_temperature`.

## 2.6 LightGBM

In [ ]:
from lightgbm import LGBMClassifier
lgbm_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", LGBMClassifier(
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
        class_weight="balanced",
    )),
])
lgbm_grid = {
    "model__n_estimators":      [200, 400],
    "model__max_depth":         [-1, 6],
    "model__learning_rate":     [0.05, 0.1],
    "model__num_leaves":        [31, 63],
    "model__min_child_samples": [20, 30],
    "model__min_child_weight":  [1e-3],
    "model__subsample":         [0.8, 1.0],
    "model__colsample_bytree":  [0.8, 1.0],
    "model__reg_alpha":         [0, 0.1],
    "model__reg_lambda":        [0, 0.1],
    "model__max_bin":           [255],
}
t0 = time.time()
lgbm_search = GridSearchCV(lgbm_pipe, lgbm_grid, scoring="recall", cv=cv,
                           n_jobs=-1, verbose=1, refit=True)
lgbm_search.fit(X_train, y_train)
print(f"Tiempo: {time.time()-t0:.1f}s")
print("Mejores params:", lgbm_search.best_params_)
lgbm_metrics, lgbm_proba = evaluate(lgbm_search.best_estimator_, X_test, y_test, "LightGBM")

**Análisis LightGBM**: extremadamente rápido (a menudo el más rápido del lote en CPU). Crece árboles por hojas (`leaf-wise`) en vez de por nivel, lo que lo hace muy preciso pero más propenso al sobreajuste si `num_leaves` es alto. Aquí lo controlamos con `min_child_samples`.

## 2.7 Comparación final

In [ ]:
results = pd.DataFrame([rf_metrics, xgb_metrics, cb_metrics, lgbm_metrics])
results = results.set_index("model").round(4)
results.style.background_gradient(cmap="Greens", axis=0)

In [ ]:
# Curvas ROC superpuestas
fig, ax = plt.subplots(figsize=(7, 6))
for name, proba in [("RandomForest", rf_proba), ("XGBoost", xgb_proba),
                    ("CatBoost", cb_proba), ("LightGBM", lgbm_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=2)

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Aleatorio")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curvas ROC — Comparación de modelos")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, search) in zip(axes, [
    ("RandomForest", rf_search), ("XGBoost", xgb_search),
    ("CatBoost", cb_search), ("LightGBM", lgbm_search),
]):
    y_pred = search.best_estimator_.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No churn", "Churn"], yticklabels=["No churn", "Churn"])
    ax.set_title(f"Confusión — {name}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

**Análisis comparativo**:
- En el problema Telco Churn lo más común es ver los cuatro modelos AUC ≈ 0.83–0.86, con diferencias pequeñas (entre 0.5 y 1.5 puntos). No esperes diferencias dramáticas.
- **Random Forest** suele ser el más conservador (recall alto si usas `class_weight='balanced'`, precision menor).
- **XGBoost / LightGBM** suelen liderar AUC, con muy poca diferencia entre ellos.
- **CatBoost** compite cabeza a cabeza, especialmente si dejas que maneje categóricas nativamente (no es nuestro caso aquí).

**Trade-offs prácticos**:
| Modelo | Velocidad train | Velocidad inferencia | Interpretabilidad | Tuning |
|---|---|---|---|---|
| RandomForest | media | media | alta | poco |
| XGBoost (GPU) | muy rápida | rápida | media | mucho |
| CatBoost | media | rápida | media | medio |
| LightGBM | muy rápida | muy rápida | media | medio |

## 2.8 Importancia de variables

Extraemos `feature_importances_` de cada modelo y graficamos el Top 10. Como el preprocesador hace one-hot, primero recuperamos los nombres expandidos de las features.

In [ ]:
def get_feature_names(pipeline):
    """Recupera los nombres de las columnas tras el ColumnTransformer."""
    pre = pipeline.named_steps["preprocessor"]
    return list(pre.get_feature_names_out())

def plot_top_features(pipeline, name, ax, top_n=10):
    feats = get_feature_names(pipeline)
    importances = pipeline.named_steps["model"].feature_importances_
    s = pd.Series(importances, index=feats).sort_values(ascending=False).head(top_n)
    s[::-1].plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"Top {top_n} features — {name}")
    ax.set_xlabel("Importancia")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_top_features(rf_search.best_estimator_,   "RandomForest", axes[0, 0])
plot_top_features(xgb_search.best_estimator_,  "XGBoost",      axes[0, 1])
plot_top_features(cb_search.best_estimator_,   "CatBoost",     axes[1, 0])
plot_top_features(lgbm_search.best_estimator_, "LightGBM",     axes[1, 1])
plt.tight_layout()
plt.show()

**Análisis**:
- Las variables con mayor peso recurrente entre los 4 modelos suelen ser **`tenure`**, **`Contract_Month-to-month`**, **`MonthlyCharges`**, **`TotalCharges`** y la combinación de servicios de internet (`InternetService_Fiber optic`, `OnlineSecurity_No`, `TechSupport_No`).
- Random Forest tiende a "diluir" la importancia entre muchas variables; XGBoost/LightGBM concentran más.
- Una variable que aparece importante para un modelo pero no para otro no siempre indica un sesgo: distintas funciones de pérdida y mecanismos de splitting reaccionan distinto a feature interactions.

**No interpretes esto como causalidad**: importance solo dice "el modelo usa esta feature para hacer splits", no "esta feature causa churn".

## 2.9 Selección y guardado del mejor modelo

In [ ]:
best = max(
    [("RandomForest", rf_search), ("XGBoost", xgb_search),
     ("CatBoost", cb_search),     ("LightGBM", lgbm_search)],
    key=lambda x: recall_score(y_test, x[1].best_estimator_.predict(X_test)),
)
best_name, best_search = best
best_pipeline = best_search.best_estimator_
print(f"Mejor modelo seleccionado: {best_name}")

In [ ]:
out_path = Path("../app/model.joblib")
out_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "model":        best_pipeline,
    "name":         best_name,
    "metrics":      results.loc[best_name].to_dict(),
    "best_params":  best_search.best_params_,
}, out_path)

print(f"Modelo guardado en {out_path.resolve()}")

**Decisión documentada**: guardamos un diccionario en lugar del Pipeline pelado. Esto nos da metadata útil en producción (`/health` puede reportar el nombre del modelo) y permite versionar comparativas si más adelante reentrenamos.

→ **Siguiente notebook**: `3_interpretability.ipynb` aplica LIME a casos individuales.